# Understanding Dual Values Well Enough to Document Them

**What this notebook is for:** helping you learn the current dual-value workflow in Sage / Passagemath well enough that you can write a small, honest section for `linear_programming.rst`.

**What you should get out of it:** not just a doc patch idea, but a working mental model of:

- what a dual value means,
- where it lives in the current stack,
- why the backend matters,
- why the docs need careful wording.

**How to use this notebook:** run it in order. Write down your own answers. When a cell asks you to explain something, do it before moving on.


In [ ]:
# Setup check -- run this first.
# If this import fails, switch to a kernel/environment where Sage works.

from sage.numerical.mip import MixedIntegerLinearProgram
print("MixedIntegerLinearProgram imported OK.")


---
## Section 1: Start with your intuition

Before touching the source, make a prediction.

Suppose you build and solve a linear program. Where would you expect to find dual values?

Possible guesses:

- directly on `MixedIntegerLinearProgram`,
- on constraint objects,
- on the backend,
- nowhere at all.

The point is not to be right. The point is to notice your assumptions before the code corrects them.


In [ ]:
my_initial_guess = {
    "where_i_expect_duals": "",
    "whether_i_expect_a_high_level_api": "",
    "whether_i_expect_this_for_generic_mips": "",
    "plain_english_meaning_of_a_dual_value": "",
}
my_initial_guess


---
## Section 2: Solve a tiny LP first

Do not begin with the docs. Begin with the object you are trying to understand.

This is a tiny linear program:

- maximize `3x + 2y`
- subject to `x + y <= 4`
- and `x <= 2`
- and `y <= 3`
- with `x, y >= 0`

This example is a little better for learning than a 2-constraint LP because it gives you three cases to think about:

- a shared resource constraint,
- an individual cap that still matters,
- and a cap that may turn out not to matter at the optimum.

Run it. Then look at the optimal solution and ask yourself what extra unit of each resource might be worth.


In [ ]:
p = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
x = p.new_variable(nonnegative=True)

p.set_objective(3*x[0] + 2*x[1])
p.add_constraint(x[0] + x[1] <= 4)
p.add_constraint(x[0] <= 2)
p.add_constraint(x[1] <= 3)

# We are interested in LP dual information, so keep the solve in simplex mode.
p.solver_parameter('simplex_or_intopt', 'simplex_only')

opt = p.solve()
vals = p.get_values(x)

print("objective:", opt)
print("variable values:", vals)


Pause here.

Without looking anything up yet, answer:

1. Which of the three constraints do you think are binding at the optimum?
2. If you were allowed one extra unit on the shared-resource inequality `x + y <= 4`, do you expect the objective to go up?
3. What about one extra unit on `x <= 2`?
4. What about one extra unit on `y <= 3`?

This is your pre-dual intuition check.


In [ ]:
pre_dual_intuition = {
    "which_constraint_feels_more_valuable": "",
    "why": "",
    "my_guess_for_shadow_prices": "",
}
pre_dual_intuition


---
## Section 3: Go find the numbers

Now ask the software where the dual values actually live.

The educational point here is boundary-tracing:

- you built the model through the frontend,
- but dual values may not live on the frontend object itself.

Inspect the backend and retrieve the row duals.


In [ ]:
b = p.get_backend()

print("backend type:", type(b))
print("row dual 0:", b.get_row_dual(0))
print("row dual 1:", b.get_row_dual(1))
print("row dual 2:", b.get_row_dual(2))


This is the first important lesson.

You did not call a generic high-level method like `p.get_dual_values()`.
You solved the model, got the backend, and then asked the backend for row duals.

That distinction should affect how you think about the docs.


In [ ]:
frontend_backend_takeaway = """
Write 2-4 sentences.

What did this teach you about where dual values live in the current stack?
Why does that matter for how a docs section should be framed?
"""
print(frontend_backend_takeaway)


---
## Section 4: Figure out which row is which

A reported number is only useful if you know what it belongs to.

The practical question is simple:

- does row `0` correspond to the first constraint added?
- does row `1` correspond to the second?
- does row `2` correspond to the third?

Use this tiny model to answer that for yourself.


In [ ]:
row_mapping_notes = {
    "what_row_0_means": "",
    "what_row_1_means": "",
    "what_row_2_means": "",
    "how_i_know": "",
}
row_mapping_notes


---
## Section 5: Verify the meaning, not just the API

This is the key experiment.

If `get_row_dual(0)` says a constraint is worth some amount, test it.

Increase that constraint's right-hand side by `1`, solve again, and compare the new objective value.

That is what turns "I can call a method" into "I understand what this number means."


In [ ]:
p2 = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
y = p2.new_variable(nonnegative=True)

p2.set_objective(3*y[0] + 2*y[1])
p2.add_constraint(y[0] + y[1] <= 5)   # first RHS increased by 1
p2.add_constraint(y[0] <= 2)
p2.add_constraint(y[1] <= 3)
p2.solver_parameter('simplex_or_intopt', 'simplex_only')

opt2 = p2.solve()
vals2 = p2.get_values(y)
b2 = p2.get_backend()

print("new objective:", opt2)
print("new variable values:", vals2)
print("new row dual 0:", b2.get_row_dual(0))
print("new row dual 1:", b2.get_row_dual(1))
print("new row dual 2:", b2.get_row_dual(2))


In [ ]:
shadow_price_check = {
    "original_objective": opt,
    "new_objective": opt2,
    "objective_change": opt2 - opt,
    "original_row_dual_0": b.get_row_dual(0),
}
shadow_price_check


Now explain the result in words.

Good target sentence shape:

- "The dual value for the first constraint is ... , which means ..."

If the objective change does not match what you expected, say why you think that is.


In [ ]:
shadow_price_explanation = """
Write 3-5 sentences explaining what you learned from the perturbation experiment.
"""
print(shadow_price_explanation)


---
## Section 6: Why the docs need careful wording

At this point you know two things:

- there is a real dual-value workflow,
- but it is accessed through the backend.

Now add the mathematical caution:

- dual values are natural LP sensitivity information,
- but you should be careful about presenting them as a generic MIP feature.

This is the second major lesson in the notebook.


In [ ]:
lp_vs_mip_reflection = """
Write 2-4 sentences.

Why should the docs talk about LP dual values carefully even though the modeling class is named MixedIntegerLinearProgram?
"""
print(lp_vs_mip_reflection)


---
## Section 7: Go read the source with better questions

Now switch to the real `passagemath` repository.

You are no longer reading the code blindly. You now know what you are looking for.

Search for the relevant implementation details:

```sh
rg -n 'get_row_dual|simplex_or_intopt|dual' src
```

And locate the docs target:

```sh
rg --files | rg 'linear_programming\.rst$'
```

Your job is to answer very specific questions now, not just wander.


In [ ]:
source_reading_notes = {
    "where_linear_programming_rst_lives": "",
    "where_get_row_dual_is_implemented": "",
    "which_backend_path_i_verified": "",
    "what_solver_mode_requirement_i_found": "",
    "what_i_found_about_row_ordering": "",
    "what_claims_i_now_feel_safe_making": "",
}
source_reading_notes


Checkpoint.

By now, you should be able to answer all of these in your own words:

1. What is the user-facing problem in the docs?
2. What is the actual workflow for getting row duals?
3. Why is this backend-specific enough to deserve careful framing?
4. What does one dual value mean in a concrete LP?

If one of those still feels fuzzy, stop and go back to either the tiny LP or the source.


---
## Section 8: Plan the smallest useful doc patch

Now you are ready to design the docs addition.

Keep the scope small. A good first patch is probably:

- one short subsection,
- one worked example,
- one note or warning block.

This is not the place to argue for a broader interface redesign.
It is the place to document an existing, useful path honestly.


In [ ]:
doc_patch_plan = {
    "where_the_new_section_should_go": "",
    "title_of_the_section": "",
    "worked_example_i_will_use": "",
    "main_interpretation_sentence": "",
    "main_warning_sentence": "",
}
doc_patch_plan


---
## Section 9: Draft the docs text

Write the smallest honest section you can.

Use this filter for every sentence:

- did I verify this in code?
- did I verify this experimentally?
- or am I only repeating something that sounds plausible?

Only keep the first two kinds.


In [ ]:
rst_draft = r'''
.. Write your candidate RST section here.
'''
print(rst_draft)


---
## Section 10: Review it like a maintainer

Read your draft and ask:

1. Does this document current behavior rather than argue for a roadmap?
2. Is it obvious that the workflow goes through the backend?
3. Would a new reader understand what the number means, not just how to print it?
4. Have I implied broader support than I actually verified?

That is the standard.


In [ ]:
maintainer_review = {
    "documents_current_behavior": "",
    "backend_scope_is_clear": "",
    "meaning_is_explained": "",
    "no_overclaiming": "",
    "final_change_i_still_need": "",
}
maintainer_review


---
## Summary

If you worked through this notebook honestly, you should now know more than "how to call `get_row_dual(i)`".

You should understand:

- where the current dual-value path lives,
- what a shadow price means in a concrete LP,
- why the backend boundary matters for documentation,
- how to write a small doc contribution that is useful without overstating support.

That is enough to make the doc patch worth doing and educational for you.
